# 🚀 Unidad 3 — Clase 4: Merge Sort

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Unidad 3, Clase 4 — Merge Sort |
| **Duración** | 50 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.  
> Ejecuta las celdas en orden de arriba hacia abajo.*

## Verificación de Dependencias

In [ ]:
import sys
required = {'numpy': 'numpy', 'matplotlib': 'matplotlib', 'ipywidgets': 'ipywidgets'}
for nombre, paquete in required.items():
    try:
        __import__(paquete)
        print(f"✅ {nombre} instalado correctamente")
    except ImportError:
        print(f"❌ {nombre} NO encontrado — instala con: pip install {paquete}")
print("\n🐍 Python", sys.version.split()[0], "| Todo listo para comenzar.")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Explicar** el paradigma Divide & Conquer y por qué garantiza O(n log n).
2. **Implementar** Merge Sort recursivo con conteo de operaciones.
3. **Demostrar** que Merge Sort es un algoritmo estable.
4. **Analizar** el costo en memoria extra O(n) y compararlo con algoritmos in-place.
5. **Comparar** Merge Sort contra los algoritmos vistos en clases anteriores.

# Sección 1: El Paradigma Divide & Conquer (8 minutos)

## ¿Por qué los algoritmos O(n²) son lentos?

En las clases anteriores vimos Selection, Insertion, Bubble, Shell y Counting Sort.  
Los primeros cuatro son O(n²) en el peor caso. ¿Cuánto importa eso?

| n | O(n²) ops | O(n log n) ops | Factor de mejora |
|---|-----------|----------------|------------------|
| 1.000 | 1.000.000 | ~10.000 | ×100 |
| 100.000 | 10.000.000.000 | ~1.700.000 | ×5.900 |
| 1.000.000 | 10¹² | ~20.000.000 | ×50.000 |

Para n=1.000.000 con una CPU que hace 10⁹ ops/seg:  
- O(n²): **~16 minutos**  
- O(n log n): **~0.02 segundos**

> 🎙️ **[PAUSA PROFESOR]** *"¿Cómo rompemos la barrera O(n²)? ¿Qué idea nueva necesitamos?"*

## La idea: Divide & Conquer

**Divide & Conquer** es un paradigma de diseño de algoritmos con tres pasos:

```
1. DIVIDE   → Partir el problema en subproblemas más pequeños
2. CONQUER  → Resolver cada subproblema recursivamente
3. COMBINE  → Fusionar las soluciones parciales
```

Para ordenamiento, la idea es:
- **DIVIDE:** Partir la lista en dos mitades
- **CONQUER:** Ordenar cada mitad (recursión)
- **COMBINE:** Fusionar (merge) dos mitades ya ordenadas

> 📌 **Intuición clave:** Fusionar dos listas ordenadas de n/2 elementos es O(n).  
> Si el árbol de recursión tiene altura log n, el costo total es O(n log n).

# Sección 2: Merge Sort — El Algoritmo (20 minutos)

## 2.1 El corazón: la función `merge`

Antes de ver el algoritmo completo, entendamos la pieza fundamental:  
**fusionar dos listas ya ordenadas en una sola lista ordenada**.

```
izq = [1, 4, 7, 9]
der = [2, 3, 8, 10]

Paso 1: comparo izq[0]=1 vs der[0]=2  → tomo 1   resultado=[1]
Paso 2: comparo izq[1]=4 vs der[0]=2  → tomo 2   resultado=[1,2]
Paso 3: comparo izq[1]=4 vs der[1]=3  → tomo 3   resultado=[1,2,3]
Paso 4: comparo izq[1]=4 vs der[2]=8  → tomo 4   resultado=[1,2,3,4]
...y así sucesivamente
```

> 💡 **¿Cuántas comparaciones hace `merge` para dos listas de n/2?**  
> A lo más n-1 comparaciones (en cada paso avanzamos al menos un puntero).

In [ ]:
# ── La función merge: fusionar dos listas ordenadas ─────────────────────────
def merge(izq: list, der: list, verbose: bool = False) -> tuple:
    """
    Fusiona dos listas ordenadas en una sola lista ordenada.

    Idea: mantener dos punteros (i, j) sobre izq y der respectivamente.
    En cada paso tomamos el menor de izq[i] y der[j] y avanzamos ese puntero.
    Cuando se agota una lista, copiamos el resto de la otra.

    Parámetros:
        izq     (list): lista izquierda ya ordenada
        der     (list): lista derecha ya ordenada
        verbose (bool): muestra cada comparación

    Retorna:
        tuple: (lista_fusionada, n_comparaciones)

    Complejidad:
        Temporal: O(n)  donde n = len(izq) + len(der)
        Espacial: O(n)  se crea una nueva lista de tamaño n
    """
    resultado = []
    i = j = 0
    comparaciones = 0

    while i < len(izq) and j < len(der):
        comparaciones += 1
        if izq[i] <= der[j]:          # <= garantiza estabilidad
            if verbose:
                print(f"  cmp: izq[{i}]={izq[i]} <= der[{j}]={der[j]} → tomo {izq[i]}")
            resultado.append(izq[i])
            i += 1
        else:
            if verbose:
                print(f"  cmp: izq[{i}]={izq[i]} >  der[{j}]={der[j]} → tomo {der[j]}")
            resultado.append(der[j])
            j += 1

    # Copiar los elementos restantes (ya están ordenados)
    resultado.extend(izq[i:])
    resultado.extend(der[j:])

    return resultado, comparaciones


# ─── Demo ──────────────────────────────────────────────────────────────────
print("=== Demo de merge ===")
izq = [1, 4, 7, 9]
der = [2, 3, 8, 10]
print(f"izq = {izq}")
print(f"der = {der}")
print()
resultado, cmp = merge(izq, der, verbose=True)
print(f"\nResultado: {resultado}")
print(f"Comparaciones usadas: {cmp} (máximo posible: {len(izq)+len(der)-1})")

## 2.2 Merge Sort completo

In [ ]:
# ── Merge Sort recursivo ────────────────────────────────────────────────────
def merge_sort(lista: list, profundidad: int = 0, verbose: bool = False) -> tuple:
    """
    Ordena 'lista' usando Merge Sort (Divide & Conquer).

    Estructura recursiva:
        1. Caso base: lista de 0 o 1 elemento → ya está ordenada
        2. Dividir: partir en mitad izquierda y mitad derecha
        3. Conquistar: ordenar cada mitad recursivamente
        4. Combinar: fusionar las dos mitades ordenadas

    Parámetros:
        lista       (list): lista a ordenar
        profundidad (int):  nivel de recursión (solo para verbose)
        verbose     (bool): traza cada nivel de recursión

    Retorna:
        tuple: (lista_ordenada, n_comparaciones)

    Complejidad:
        Temporal: O(n log n) en todos los casos (mejor, promedio y peor)
        Espacial: O(n) extra — no es in-place
        Estabilidad: estable (merge usa <=)
    """
    sangria = "  " * profundidad

    # ── Caso base ──────────────────────────────────────────────────────────
    if len(lista) <= 1:
        if verbose:
            print(f"{sangria}BASE {lista}")
        return lista[:], 0

    # ── Dividir ────────────────────────────────────────────────────────────
    medio = len(lista) // 2
    izq_orig = lista[:medio]
    der_orig = lista[medio:]

    if verbose:
        print(f"{sangria}DIVIDE {lista} → {izq_orig} | {der_orig}")

    # ── Conquistar ─────────────────────────────────────────────────────────
    izq_ord, cmp_izq = merge_sort(izq_orig, profundidad + 1, verbose)
    der_ord, cmp_der = merge_sort(der_orig, profundidad + 1, verbose)

    # ── Combinar ───────────────────────────────────────────────────────────
    resultado, cmp_merge = merge(izq_ord, der_ord)

    if verbose:
        print(f"{sangria}MERGE  {izq_ord} + {der_ord} → {resultado}")

    return resultado, cmp_izq + cmp_der + cmp_merge


# ─── Demo ──────────────────────────────────────────────────────────────────
print("=== Merge Sort — traza completa ===")
datos = [5, 2, 8, 1, 9, 3]
print(f"Entrada: {datos}\n")
resultado, total_cmp = merge_sort(datos, verbose=True)
print(f"\nResultado:     {resultado}")
print(f"Comparaciones: {total_cmp}")
print(f"n log₂ n ≈     {len(datos) * (len(datos).bit_length() - 1):.0f}  (cota teórica)")

## 2.3 Visualización del árbol de recursión

In [ ]:
# Visualización del árbol de recursión de Merge Sort
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def construir_arbol(lista, nodo_id=0, nivel=0, nodos=None, aristas=None):
    """Construye el árbol de recursión como listas de nodos y aristas."""
    if nodos is None: nodos = []; aristas = []
    nodos.append({'id': nodo_id, 'lista': lista[:], 'nivel': nivel})
    if len(lista) <= 1:
        return nodos, aristas, nodo_id
    medio = len(lista) // 2
    id_izq = nodo_id * 2 + 1
    id_der = nodo_id * 2 + 2
    aristas.append((nodo_id, id_izq))
    aristas.append((nodo_id, id_der))
    construir_arbol(lista[:medio], id_izq, nivel + 1, nodos, aristas)
    construir_arbol(lista[medio:], id_der, nivel + 1, nodos, aristas)
    return nodos, aristas, nodo_id

def posicion_nodo(nodo_id, nivel, max_nivel):
    """Calcula posición x,y para un nodo del árbol."""
    pos_en_nivel = nodo_id - (2**nivel - 1)
    total_en_nivel = 2**nivel
    x = (pos_en_nivel + 0.5) / total_en_nivel
    y = 1.0 - nivel / (max_nivel + 1)
    return x, y

datos_arbol = [8, 3, 1, 5, 2, 7, 4, 6]
nodos, aristas, _ = construir_arbol(datos_arbol)
max_nivel = max(n['nivel'] for n in nodos)

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 1); ax.set_ylim(-0.05, 1.05)
ax.axis('off')
ax.set_title('Árbol de Recursión — Merge Sort\n(cada nivel procesa n elementos → costo total O(n log n))',
             fontsize=12, fontweight='bold')

colores_nivel = ['#1565C0', '#2196F3', '#64B5F6', '#BBDEFB', '#E3F2FD']

# Calcular posiciones
pos = {}
for nodo in nodos:
    nid, nivel = nodo['id'], nodo['nivel']
    x, y = posicion_nodo(nid, nivel, max_nivel)
    pos[nid] = (x, y)

# Dibujar aristas
for (padre, hijo) in aristas:
    if padre in pos and hijo in pos:
        xp, yp = pos[padre]; xh, yh = pos[hijo]
        ax.plot([xp, xh], [yp, yh], 'k-', alpha=0.3, linewidth=1)

# Dibujar nodos
for nodo in nodos:
    nid, nivel, lst = nodo['id'], nodo['nivel'], nodo['lista']
    if nid not in pos: continue
    x, y = pos[nid]
    color = colores_nivel[min(nivel, len(colores_nivel)-1)]
    etiqueta = str(lst)
    fontsize = max(6, 9 - nivel)
    ax.text(x, y, etiqueta, ha='center', va='center', fontsize=fontsize,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.85,
                      edgecolor='white', linewidth=1.5), color='white', fontweight='bold')

# Leyenda de niveles
for i in range(min(max_nivel+1, len(colores_nivel))):
    n_en_nivel = min(2**i, len(datos_arbol))
    ax.text(0.01, 1.0 - i/(max_nivel+1), f' Nivel {i}: {n_en_nivel} subproblemas',
            fontsize=8, va='center', color=colores_nivel[i], fontweight='bold')

plt.tight_layout()
plt.show()
print(f"n={len(datos_arbol)}, niveles del árbol={max_nivel+1}, log₂({len(datos_arbol)})≈{np.log2(len(datos_arbol)):.1f}")

# Sección 3: Animación Interactiva (10 minutos)

In [ ]:
# Animación de Merge Sort — muestra las fases de merge en orden
import matplotlib.animation as animation
from IPython.display import HTML, display

try:
    import google.colab; EN_COLAB = True
except ImportError:
    EN_COLAB = False

if not EN_COLAB:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')
    except Exception:
        get_ipython().run_line_magic('matplotlib', 'inline')

def capturar_frames_merge_sort(lista):
    """Ejecuta Merge Sort capturando el estado del arreglo en cada operación."""
    frames = []
    n = len(lista)
    trabajo = lista[:]

    def _merge_sort_frames(arr, inicio):
        if len(arr) <= 1:
            return arr
        medio = len(arr) // 2
        izq = _merge_sort_frames(arr[:medio], inicio)
        der = _merge_sort_frames(arr[medio:], inicio + medio)

        # Merge con capturas
        i = j = 0
        res = []
        rango = list(range(inicio, inicio + len(arr)))
        while i < len(izq) and j < len(der):
            if izq[i] <= der[j]:
                res.append(izq[i]); i += 1
            else:
                res.append(der[j]); j += 1
            # Actualizar el estado global
            for k, v in enumerate(res):
                trabajo[rango[k]] = v
            frames.append((trabajo[:], list(rango[:len(res)]),
                           rango[0], rango[-1]))
        res.extend(izq[i:]); res.extend(der[j:])
        for k, v in enumerate(res):
            trabajo[rango[k]] = v
        frames.append((trabajo[:], rango, rango[0], rango[-1]))
        return res

    frames.append((lista[:], [], 0, n-1))
    _merge_sort_frames(lista[:], 0)
    frames.append((trabajo[:], list(range(n)), 0, n-1))
    return frames

def animar_merge_sort(datos_orig, interval=300):
    frames = capturar_frames_merge_sort(datos_orig)
    n = len(datos_orig)

    COLOR_BASE    = '#90CAF9'
    COLOR_ACTIVO  = '#FF7043'
    COLOR_HECHO   = '#A5D6A7'

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(0, max(datos_orig) + 3)
    ax.set_xlabel('Índice'); ax.set_ylabel('Valor')
    ax.grid(axis='y', alpha=0.3)

    estado0, _, _, _ = frames[0]
    bars   = ax.bar(range(n), estado0, color=COLOR_BASE, edgecolor='white', linewidth=1.2)
    textos = [ax.text(i, estado0[i] + 0.3, str(estado0[i]),
                      ha='center', va='bottom', fontsize=8) for i in range(n)]

    def actualizar(fidx):
        estado, activos, inicio, fin = frames[fidx]
        es_final = (fidx == len(frames) - 1)
        for i, (bar, txt) in enumerate(zip(bars, textos)):
            bar.set_height(estado[i])
            txt.set_position((i, estado[i] + 0.3))
            txt.set_text(str(estado[i]))
            if es_final:
                bar.set_color(COLOR_HECHO)
            elif i in activos:
                bar.set_color(COLOR_ACTIVO)
            elif inicio <= i <= fin:
                bar.set_color('#FFB74D')
            else:
                bar.set_color(COLOR_BASE)
        titulo = 'Merge Sort — ¡Ordenado!' if es_final else f'Merge Sort — frame {fidx}'
        ax.set_title(titulo, fontsize=12, fontweight='bold')
        return bars

    anim = animation.FuncAnimation(fig, actualizar, frames=len(frames),
                                   interval=interval, blit=False, repeat=False)
    plt.tight_layout()
    display(HTML(anim.to_jshtml()))
    plt.close()

import random; random.seed(42)
datos_demo = random.sample(range(1, 21), 10)
print(f"Datos: {datos_demo}")
animar_merge_sort(datos_demo, interval=250)

# Sección 4: Análisis de Complejidad (7 minutos)

## 4.1 Ecuación de recurrencia

Sea T(n) el número de comparaciones que hace Merge Sort sobre una lista de n elementos:

$$T(n) = \underbrace{2 \cdot T\left(\frac{n}{2}\right)}_{\text{dos subproblemas}} + \underbrace{O(n)}_{\text{merge}}$$

Con caso base T(1) = 0.

## 4.2 Solución por el Teorema Maestro

La ecuación tiene la forma $T(n) = aT(n/b) + f(n)$ con **a=2, b=2, f(n)=O(n)**:

$$n^{\log_b a} = n^{\log_2 2} = n^1 = n$$

Como $f(n) = \Theta(n^{\log_b a})$, estamos en el **Caso 2 del Teorema Maestro**:

$$T(n) = \Theta(n \log n)$$

## 4.3 Intuición geométrica

```
Nivel 0:  [n elementos] ───────── 1 merge de costo n    = n
Nivel 1:  [n/2][n/2]   ───────── 2 merges de costo n/2 = n
Nivel 2:  [n/4]×4      ───────── 4 merges de costo n/4 = n
...
Nivel k:  [1]×n        ───────── n merges de costo 1   = n

Total niveles: log₂ n
Costo por nivel: n
───────────────────────────────────────────────
Total: n × log₂ n = O(n log n)
```

> 📌 **Propiedad notable:** Merge Sort tiene la **misma complejidad** en mejor, promedio y peor caso.  
> A diferencia de Quick Sort (que veremos después), Merge Sort no tiene peor caso O(n²).

## 4.4 Costo en memoria

Merge Sort **no es in-place**: necesita O(n) memoria extra para el merge.  
La pila de recursión ocupa O(log n) adicional.

| Algoritmo | Tiempo (peor) | Memoria extra | In-place |
|-----------|--------------|---------------|----------|
| Insertion Sort | O(n²) | O(1) | ✅ Sí |
| Shell Sort | O(n^1.5) | O(1) | ✅ Sí |
| Counting Sort | O(n+k) | O(k) | ❌ No |
| **Merge Sort** | **O(n log n)** | **O(n)** | **❌ No** |

> 🎙️ **[PAUSA PROFESOR]** *"¿En qué situaciones el costo de memoria O(n) podría ser problemático?"*

In [ ]:
# Verificación empírica de la complejidad O(n log n)
import timeit, random, math
import matplotlib.pyplot as plt
import numpy as np

def insertion_sort_ref(lista):
    a = lista[:]
    for i in range(1, len(a)):
        c = a[i]; j = i - 1
        while j >= 0 and a[j] > c: a[j+1] = a[j]; j -= 1
        a[j+1] = c
    return a

ns     = [100, 500, 1000, 2000, 5000, 10000]
t_ms   = []
t_ins  = []
cmp_ms = []

for n in ns:
    datos = random.sample(range(n * 3), n)
    reps  = max(5, 500 // n)
    t_ms.append(timeit.timeit(lambda: merge_sort(datos[:]), number=reps) / reps * 1000)
    t_ins.append(timeit.timeit(lambda: insertion_sort_ref(datos[:]), number=reps) / reps * 1000)
    _, cmp = merge_sort(datos[:])
    cmp_ms.append(cmp)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Tiempo
ax1.plot(ns, t_ms,  'o-', color='#1565C0', linewidth=2, label='Merge Sort')
ax1.plot(ns, t_ins, 's--', color='#E53935', linewidth=2, label='Insertion Sort')
nlogn = [n * math.log2(n) * (t_ms[-1] / (ns[-1] * math.log2(ns[-1]))) for n in ns]
ax1.plot(ns, nlogn, ':', color='gray', linewidth=1.5, label='O(n log n) ref.')
ax1.set_xlabel('n'); ax1.set_ylabel('Tiempo (ms)')
ax1.set_title('Tiempo: Merge Sort vs Insertion Sort')
ax1.legend(); ax1.grid(alpha=0.3)

# Comparaciones vs n log n teórico
teorico = [n * math.log2(n) for n in ns]
ratio   = [c / t for c, t in zip(cmp_ms, teorico)]
ax2.bar(range(len(ns)), ratio, color='#1565C0', alpha=0.8)
ax2.axhline(y=sum(ratio)/len(ratio), color='red', linestyle='--',
            label=f'Promedio ≈ {sum(ratio)/len(ratio):.2f}')
ax2.set_xticks(range(len(ns))); ax2.set_xticklabels(ns)
ax2.set_xlabel('n'); ax2.set_ylabel('comparaciones / (n log₂ n)')
ax2.set_title('Comparaciones reales vs teórico n log₂ n')
ax2.legend(); ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"{'n':>6} | {'Merge Sort':>12} | {'Insertion':>12} | {'Factor mejora':>14}")
print("-" * 52)
for n, tm, ti in zip(ns, t_ms, t_ins):
    factor = ti / tm if tm > 0 else float('inf')
    print(f"{n:>6} | {tm:>10.3f}ms | {ti:>10.3f}ms | {factor:>12.1f}×")

# Sección 5: Estabilidad de Merge Sort (5 minutos)

## ¿Qué significa ser estable?

Un algoritmo de ordenamiento es **estable** si mantiene el orden relativo  
de elementos con la misma clave.

**Ejemplo:** ordenar personas por edad:
```
Entrada:  [("Ana", 25), ("Carlos", 30), ("Beatriz", 25), ("David", 20)]
                                           ↑ misma edad que Ana

Algoritmo ESTABLE:     [David/20, Ana/25, Beatriz/25, Carlos/30]  ← Ana antes que Beatriz ✅
Algoritmo INESTABLE:   [David/20, Beatriz/25, Ana/25, Carlos/30]  ← orden cambiado ❌
```

Merge Sort es estable porque en `merge` usamos `<=` (tomamos del izquierdo primero  
cuando hay empate), preservando el orden original.

In [ ]:
# Demostración de estabilidad de Merge Sort

# Adaptamos merge_sort para trabajar con tuplas, comparando solo la clave
def merge_estable(izq, der, clave=lambda x: x):
    resultado = []
    i = j = 0
    while i < len(izq) and j < len(der):
        # <= garantiza estabilidad: ante empate tomamos del izquierdo (más antiguo)
        if clave(izq[i]) <= clave(der[j]):
            resultado.append(izq[i]); i += 1
        else:
            resultado.append(der[j]); j += 1
    resultado.extend(izq[i:]); resultado.extend(der[j:])
    return resultado

def merge_sort_estable(lista, clave=lambda x: x):
    if len(lista) <= 1: return lista[:]
    medio = len(lista) // 2
    izq = merge_sort_estable(lista[:medio], clave)
    der = merge_sort_estable(lista[medio:], clave)
    return merge_estable(izq, der, clave)

personas = [
    ("Ana",     25),
    ("Carlos",  30),
    ("Beatriz", 25),
    ("David",   20),
    ("Elena",   30),
    ("Felipe",  20),
]

ordenado = merge_sort_estable(personas, clave=lambda p: p[1])

print("=== Demostración de Estabilidad ===")
print(f"{'Entrada:':<12} {[f'{n}/{e}' for n, e in personas]}")
print(f"{'Ordenado:':<12} {[f'{n}/{e}' for n, e in ordenado]}")
print()
print("Verificación por grupo de edad:")
from itertools import groupby
for edad, grupo in groupby(ordenado, key=lambda p: p[1]):
    g = list(grupo)
    idx_orig = [personas.index(p) for p in g]
    es_estable = idx_orig == sorted(idx_orig)
    print(f"  Edad {edad}: {[n for n,_ in g]} | orden original preservado: {'✅' if es_estable else '❌'}")

# Sección 6: Widget Interactivo — Explorador de Merge Sort

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import random, timeit, math

slider_n   = widgets.IntSlider(value=16, min=4, max=128, step=4,
                                description='n:', style={'description_width': '120px'})
tipo_datos = widgets.Dropdown(
    options=[('Aleatorio', 'random'), ('Casi ordenado', 'nearly'),
             ('Invertido', 'reversed'), ('Ya ordenado', 'sorted')],
    value='random', description='Datos:', style={'description_width': '120px'})
boton  = widgets.Button(description='▶ Ejecutar', button_style='primary')
salida = widgets.Output()

def al_ejecutar(b):
    with salida:
        salida.clear_output(wait=True)
        n    = slider_n.value
        tipo = tipo_datos.value

        if tipo == 'random':   datos = random.sample(range(n * 3), n)
        elif tipo == 'nearly':
            datos = list(range(n))
            for _ in range(max(1, n // 10)):
                i, j = random.randint(0, n-2), random.randint(0, n-2)
                datos[i], datos[j] = datos[j], datos[i]
        elif tipo == 'reversed': datos = list(range(n, 0, -1))
        else:                    datos = list(range(n))

        resultado, cmp = merge_sort(datos[:])
        t = timeit.timeit(lambda: merge_sort(datos[:]), number=200) / 200

        teorico_nlogn = n * math.log2(n) if n > 1 else 0
        print(f"n={n} | tipo={tipo}")
        print(f"Comparaciones reales: {cmp}")
        print(f"n·log₂(n) teórico:    {teorico_nlogn:.1f}")
        print(f"Ratio real/teórico:   {cmp/teorico_nlogn:.3f}" if teorico_nlogn else "")
        print(f"Tiempo:               {t*1000:.4f} ms")
        print(f"¿Correcto?            {'✅' if resultado == sorted(datos) else '❌'}")

boton.on_click(al_ejecutar)
display(widgets.VBox([
    widgets.HBox([slider_n, tipo_datos]),
    boton, salida
]))

# Sección 7: Ejercicios Prácticos

## 🧪 Ejercicio 1 ⭐: Trazar Merge Sort a mano

Aplica Merge Sort a la lista `[6, 3, 8, 2, 9, 1, 7, 4]` **sin ejecutar código**.  
Dibuja el árbol de recursión mostrando:
- División en cada nivel
- Listas ordenadas al subir (merge)
- Total de comparaciones

Luego ejecuta la celda para verificar tu respuesta.

In [ ]:
# Verificación del Ejercicio 1
datos_ej1 = [6, 3, 8, 2, 9, 1, 7, 4]
resultado, cmp = merge_sort(datos_ej1, verbose=True)
print(f"\nResultado final: {resultado}")
print(f"Total comparaciones: {cmp}")
print(f"n·log₂(n) = {len(datos_ej1) * math.log2(len(datos_ej1)):.1f}")

## 🧪 Ejercicio 2 ⭐: Merge de k listas ordenadas

**Descripción:** Implementa `merge_k_listas(listas)` que fusiona k listas ordenadas  
en una sola lista ordenada usando la función `merge` ya definida.

**Ejemplo:**
```
Entrada:  [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
Salida:   [1, 2, 3, 4, 5, 6, 7, 8, 9]
```

**Sugerencia:** Aplica `merge` sucesivamente a pares de listas.

In [ ]:
def merge_k_listas(listas: list) -> list:
    """
    Fusiona k listas ordenadas en una sola lista ordenada.

    Parámetros:
        listas (list): lista de listas, cada una ordenada ascendentemente
    Retorna:
        list: lista fusionada y ordenada
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_2(fn):
    import time
    casos = [
        ([[1, 4, 7], [2, 5, 8], [3, 6, 9]],     [1,2,3,4,5,6,7,8,9], "3 listas de 3"),
        ([[1, 2, 3]],                             [1, 2, 3],           "1 lista"),
        ([[], [1, 2], []],                        [1, 2],              "Listas vacías"),
        ([],                                      [],                  "Sin listas"),
        ([[5], [3], [8], [1]],                    [1, 3, 5, 8],        "4 listas de 1 elemento"),
        ([[1,3,5,7,9], [2,4,6,8,10]],             list(range(1,11)),   "2 listas de 5"),
        ([[i*3+1, i*3+2, i*3+3] for i in range(4)],
         sorted([i*3+j for i in range(4) for j in [1,2,3]]),          "4 listas solapadas"),
    ]
    aprobados = 0
    for listas, esperado, desc in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn([l[:] for l in listas])
            t1 = time.perf_counter()
            if resultado == esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ {desc}")
                print(f"     Esperado: {esperado}")
                print(f"     Obtenido: {resultado}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_2(merge_k_listas)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def merge_k_listas(listas: list) -> list:
#     """Fusiona k listas usando merge sucesivo."""
#     if not listas:
#         return []
#     resultado = listas[0][:]
#     for i in range(1, len(listas)):
#         resultado, _ = merge(resultado, listas[i][:])
#     return resultado
#
# # Nota: este enfoque es O(k·n) en total.
# # Una implementación óptima con heap es O(N log k) donde N = total de elementos.

## 🧪 Ejercicio 3 ⭐⭐: Contar inversiones

**Descripción:** Implementa `contar_inversiones(lista)` que retorna el número de pares  
(i, j) con i < j y lista[i] > lista[j] usando una modificación de Merge Sort.

**Ejemplo:**
```
lista = [3, 1, 2]
Inversiones: (3,1), (3,2) → 2 inversiones
```

**Restricciones:** Debe ser O(n log n), **no** O(n²).

**Pista:** Durante el merge, cuando tomamos un elemento de `der` antes que de `izq`,  
todos los elementos restantes de `izq` forman inversiones con él.

In [ ]:
def contar_inversiones(lista: list) -> int:
    """
    Cuenta el número de inversiones en 'lista' usando Merge Sort modificado.

    Una inversión es un par (i, j) con i < j y lista[i] > lista[j].

    Parámetros:
        lista (list): lista de enteros
    Retorna:
        int: número de inversiones

    Complejidad esperada: O(n log n)
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_3(fn):
    import time
    def contar_fuerza_bruta(lst):
        return sum(1 for i in range(len(lst)) for j in range(i+1, len(lst)) if lst[i] > lst[j])

    casos = [
        ([3, 1, 2],        2,  "Ejemplo básico"),
        ([1, 2, 3, 4],     0,  "Ya ordenada → 0 inversiones"),
        ([4, 3, 2, 1],     6,  "Invertida → n(n-1)/2 inversiones"),
        ([],               0,  "Lista vacía"),
        ([1],              0,  "Un elemento"),
        ([2, 2, 2],        0,  "Todos iguales"),
        ([5, 1, 4, 2, 3],  6,  "Ejemplo mixto"),
    ]
    import random; random.seed(99)
    for _ in range(3):
        lst = random.sample(range(50), 15)
        casos.append((lst, contar_fuerza_bruta(lst), f"Aleatorio n=15"))

    aprobados = 0
    for lista, esperado, desc in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn(lista[:])
            t1 = time.perf_counter()
            if resultado == esperado:
                print(f"  ✅ {desc} — {resultado} inversiones ({(t1-t0)*1000:.2f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ {desc} | Esperado: {esperado} | Obtenido: {resultado}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_3(contar_inversiones)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def contar_inversiones(lista: list) -> int:
#     """Cuenta inversiones modificando Merge Sort. O(n log n)."""
#     def _merge_count(arr):
#         if len(arr) <= 1:
#             return arr[:], 0
#         medio = len(arr) // 2
#         izq, inv_izq = _merge_count(arr[:medio])
#         der, inv_der = _merge_count(arr[medio:])
#
#         # Fusionar y contar
#         resultado = []; inv = inv_izq + inv_der
#         i = j = 0
#         while i < len(izq) and j < len(der):
#             if izq[i] <= der[j]:
#                 resultado.append(izq[i]); i += 1
#             else:
#                 # Todos los izq[i..] > der[j] forman inversiones con der[j]
#                 resultado.append(der[j]); j += 1
#                 inv += len(izq) - i
#         resultado.extend(izq[i:]); resultado.extend(der[j:])
#         return resultado, inv
#
#     _, total = _merge_count(lista)
#     return total

## 🔬 Zona de Experimentación

Sugerencias:
- ¿Qué pasa si en `merge` cambias `<=` por `<`? ¿Sigue siendo estable?
- Implementa Merge Sort iterativo (bottom-up) sin recursión
- ¿Cuándo es conveniente usar Insertion Sort para subproblemas pequeños? (Timsort)

In [ ]:
# Espacio libre para experimentar

In [ ]:
# Espacio libre para experimentar